In [17]:
import json

log_list = []
# with open("/Users/seojune/Desktop/agent_hard_benchmark/ComplexFuncBench/data/ComplexFuncBench_assessed.jsonl", "r") as f:
with open("/Users/seojune/Desktop/agent_hard_benchmark/ComplexFuncBench/llm-filtering/results/ComplexFuncBench_assessed-gpt4_1.jsonl", "r") as f:
    for line in f:
        log_list.append(json.loads(line))

In [ ]:
incorrect_ids_llm = []
count = 0

from collections import defaultdict
group_count = defaultdict(int)

with open("incorrect_ids_manual.txt", "r") as f:
    incorrect_ids_manual = [line.strip() for line in f.readlines() if line[0] != '#']
    incorrect_ids_manual = list(set(incorrect_ids_manual))

for log in log_list:
    is_flawed = log['assessment']['is_flawed']
    error_category = log['assessment'].get('error_category', '')
    reasoning = log['assessment'].get('reasoning_summary', '')
    sample_id = log['id']
    if is_flawed:
        incorrect_ids_llm.append(sample_id)
        count += 1
        group_count[sample_id[:-1]] += 1

        if sample_id not in incorrect_ids_manual:
            print(sample_id, ":", error_category)
            print(reasoning, "\n")
print(f"Total samples with is_flawed = true: {count}")

Car-Rental-2 : Argument Value Mismatch
The Search_Car_Rentals calls used the coordinates for 'Hilton San Francisco Union Square' instead of 'San Francisco - Ellis Street,' which contradicts the user's explicit request. 

Car-Rental-21 : Dataset Integrity Issue
The drop-off coordinates in the 'Search_Car_Rentals' call use the city center of San Jose, but the dataset only supports drop-off at Norman Y. Mineta San Jose International Airport. 

Car-Rental-25 : Dataset Integrity Issue
The Search_Car_Rentals call uses Newark city coordinates for drop-off, but only airport drop-off is available in the dataset, making the function call unexecutable as intended. 

Car-Rental-101 : Argument Value Mismatch
The assistant proceeds with a car rental result that drops off at 'Alexandria Downtown' instead of 'Sydney Kingsford Smith Airport', contradicting the user's explicit request. 

Car-Rental-121 : Argument Value Mismatch
The drop-off location in the car rental search is set to Eindhoven Airport i

In [ ]:
# set으로 변환
manual_set = set(incorrect_ids_manual)
llm_set = set(incorrect_ids_llm)

# True Positive, False Positive, False Negative
tp = len(manual_set & llm_set)
fp = len(llm_set - manual_set)
fn = len(manual_set - llm_set)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")

Precision: 0.236
Recall: 0.833
F1-score: 0.367


In [18]:
with open("/Users/seojune/Desktop/agent_hard_benchmark/ComplexFuncBench/llm-filtering/incorrect_ids_llm.txt", "w") as f:
    for sample_id in llm_set:
        f.write(f"{sample_id}\n")

In [ ]:
# GPT 4.1: 

Precision: 0.236
Recall: 0.833
F1-score: 0.367

# GPT 4o-20240806:
Precision: 0.190
Recall: 0.574
F1-score: 0.286